In [ ]:
"""
PRS分数与连续表型的线性回归分析
遍历 all_PRS_result 目录下的所有 sum_score_*.txt 文件，
提取clump参数、P值阈值和表型名称，
对每个文件计算线性回归（score -> 连续表型），
输出结果表格，并绘制按R2分组的Beta和P值柱状图。
"""

import pandas as pd
import numpy as np
import statsmodels.api as sm
from pathlib import Path
import matplotlib.pyplot as plt
import warnings
import re
warnings.filterwarnings('ignore')

# ======================== 配置参数 ========================
PRS_ROOT = "all_PRS_result"          # PRS结果的根目录
PHENO_FILE = r"pheno.txt"   # 连续表型文件：至少包含IID和表型值两列
PHENO_COL = "phenotype"              # 表型文件中表型值的列名（如果文件有表头），否则设为None
OUTPUT_CSV = "prs_linear_results.csv"
OUTPUT_excel = "prs_linear_results.xlsx"
BETA_PLOT = "beta_barplot_by_r2.png"
P_PLOT = "pvalue_barplot_by_r2.png"
Trait_Name = 'PRS'            # 用于图标题的自定义表型名称
Trait_manual = True                  # 是否使用自定义表型名称

# ======================== 辅助函数 ========================
def load_pheno(pheno_file_path, pheno_col=None):
    """
    加载连续表型文件，返回字典 {IID: phenotype_value}
    文件格式：若pheno_col为None，则无表头，两列：IID  phenotype
             若pheno_col指定列名，则文件有表头，从该列读取表型值
    """
    if pheno_col is None:
        # 无表头，两列：IID, phenotype
        df = pd.read_csv(pheno_file_path, sep='\s+', header=None, names=['IID','FID', 'phenotype'])
    else:
        df = pd.read_csv(pheno_file_path, sep='\s+')
        if pheno_col not in df.columns:
            raise ValueError(f"表型文件中未找到列 '{pheno_col}'，可用列：{df.columns.tolist()}")
    df=df[df['phenotype']<7.001]
    # 确保表型数值型，丢弃缺失
    df['phenotype'] = pd.to_numeric(df['phenotype'], errors='coerce')
    df = df.dropna(subset=['phenotype'])
    pheno_dict = dict(zip(df['IID'], df['phenotype']))
    print(f"加载了 {len(pheno_dict)} 个个体的表型值")
    return pheno_dict

def parse_file_info(file_path):
    """
    根据文件路径解析参数：
    all_PRS_result / clump_param / p_thresh / sum_score_trait.txt
    返回 (clump_param, p_thresh, trait)
    """
    path = Path(file_path)
    trait = path.stem.replace("sum_score_", "")   # 去掉前缀 sum_score_
    p_thresh = float(path.parent.name.replace('p_',''))                   # 如 p_5e-06
    clump_param = path.parent.parent.name         # 如 clump_1000_r2_0.001
    return clump_param, p_thresh, trait

def extract_r2(clump_param):
    """从clump_param字符串中提取R2值，如 'clump_1000_r2_0.001' -> 0.001"""
    match = re.search(r'[rR]2_([0-9.]+)', clump_param)
    if match:
        return float(match.group(1))
    else:
        # 若提取失败，使用整个字符串作为分组键（但绘图时需转为字符串）
        return clump_param

def perform_linear_regression(df, score_col='score', pheno_col='phenotype'):
    """
    对给定DataFrame进行线性回归: phenotype ~ score
    返回 beta, p_value, std_err, n
    若回归失败则返回 None
    """
    y = df[pheno_col]
    X = df[score_col]
    X_with_const = sm.add_constant(X)
    try:
        model = sm.OLS(y, X_with_const)
        result = model.fit()
        beta = result.params[score_col]
        p_value = result.pvalues[score_col]
        std_err = result.bse[score_col]
        n = len(df)
        return beta, p_value, std_err, n
    except Exception as e:
        print(f"线性回归失败: {e}")
        return None

def format_p_thresh_label(p_thresh):
    """将 'p_5e-06' 格式化为 '5e-06' 用于X轴标签"""
    if p_thresh.startswith('p_'):
        return p_thresh[2:]
    else:
        return p_thresh

# ======================== 主流程 ========================
def main():
    # 1. 读取连续表型字典
    print("加载连续表型数据...")
    pheno_dict = load_pheno(PHENO_FILE, None)

    # 2. 遍历所有sum_score文件
    score_files = list(Path(PRS_ROOT).glob("*/*/sum_score_*.txt"))
    print(f"找到 {len(score_files)} 个分数文件")

    results = []
    for file_path in score_files:
        print(f"处理: {file_path}")
        clump, pthresh, trait = parse_file_info(file_path)
        try:
            score_df = pd.read_csv(file_path, sep='\s+')
            if 'IID' not in score_df.columns or 'score' not in score_df.columns:
                print(f"  跳过：文件缺少 IID 或 score 列")
                continue

            # 添加表型列
            score_df['phenotype'] = score_df['IID'].map(pheno_dict)
            score_df = score_df.dropna(subset=['score', 'phenotype'])
            if score_df.empty:
                print(f"  跳过：无有效样本")
                continue

            n_total = len(score_df)
            if n_total == 0:
                print(f"  跳过：样本量为0")
                continue

            reg_result = perform_linear_regression(score_df)
            if reg_result is None:
                print(f"  跳过：回归失败")
                continue
            beta, p_val, std_err, n = reg_result

            # 提取R2值
            r2_val = extract_r2(clump)

            results.append({
                'clump_param': clump,
                'r2': r2_val,
                'p_thresh': pthresh,
                'trait': trait,
                'N': n,
                'beta': beta,
                'std_err': std_err,
                'p_value': p_val
            })
            print(f"  完成: beta={beta:.4f}, p={p_val:.2e}, N={n}")
        except Exception as e:
            print(f"  错误: {e}")
            continue

    if not results:
        print("没有成功计算任何结果，程序退出。")
        return

    # 3. 保存结果表格
    result_df = pd.DataFrame(results)
    result_df.to_csv(OUTPUT_CSV, index=False)
    result_df.to_excel(OUTPUT_excel, index=False)
    print(f"结果已保存至 {OUTPUT_excel}")

    # 检查trait是否一致
    unique_traits = result_df['trait'].unique()
    if len(unique_traits) > 1:
        print(f"警告：检测到多个不同的trait ({unique_traits})，将使用第一个作为suptitle")
        main_trait = unique_traits[0]
    else:
        main_trait = unique_traits[0]
    if Trait_manual:
        main_trait = Trait_Name

    # 4. 按R2分组绘图
    # 确保r2数值排序正确（如果是数值型，则排序，否则按字符串）
    if all(isinstance(x, (int, float)) for x in result_df['r2']):
        r2_groups = sorted(result_df['r2'].unique())
    else:
        r2_groups = sorted(result_df['r2'].unique(), key=str)

    n_groups = len(r2_groups)
    ncols = min(3, n_groups)
    nrows = (n_groups + ncols - 1) // ncols

    # ------------------ Beta 柱状图（按R2子图） ------------------
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows), squeeze=False, dpi=600)
    fig.suptitle(f'Linear Regression Beta for PRS Score\nTrait: {main_trait}', fontsize=16)

    for idx, r2_val in enumerate(r2_groups):
        row = idx // ncols
        col = idx % ncols
        ax = axes[row, col]
        sub_df = result_df[result_df['r2'] == r2_val].copy()
        # 按p_thresh排序（确保顺序一致）
        sub_df = sub_df.sort_values('p_thresh')
        p_labels = [('%.0e' % p).replace('e', "E") for p in sub_df['p_thresh']]
        x = np.arange(len(sub_df))
        bars = ax.bar(x, sub_df['beta'], color='steelblue', edgecolor='black')
        ax.set_xticks(x)
        ax.set_xticklabels(p_labels, rotation=45, ha='right', fontsize=8)
        ax.set_ylabel('Beta Coefficient')
        ax.set_title(f'R² = {r2_val}')
        ax.set_xlabel('P threshold')
        ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)

        # 标注beta值（两位小数）
        for bar, beta_val in zip(bars, sub_df['beta']):
            height = bar.get_height()
            if height >= 0:
                y_pos = height + 0.01 * (sub_df['beta'].max() - sub_df['beta'].min()) if len(sub_df) > 1 else height + 0.01
            else:
                y_pos = height - 0.01 * (sub_df['beta'].max() - sub_df['beta'].min()) if len(sub_df) > 1 else height - 0.01
            ax.text(bar.get_x() + bar.get_width()/2., y_pos,
                    f'{beta_val:.2f}', ha='center', va='bottom' if height >= 0 else 'top',
                    fontsize=10, color='black')

    # 隐藏多余的子图
    for idx in range(n_groups, nrows*ncols):
        row = idx // ncols
        col = idx % ncols
        axes[row, col].axis('off')

    plt.tight_layout()
    plt.subplots_adjust(top=0.8)
    plt.savefig(BETA_PLOT, dpi=300)
    plt.close()
    print(f"Beta柱状图（按R2分组）已保存至 {BETA_PLOT}")

    # ------------------ P值柱状图（-log10(P)，按R2子图） ------------------
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows), squeeze=False, dpi=600)
    fig.suptitle(f'Significance of PRS Score (Linear Regression)\nTrait: {main_trait}', fontsize=16)

    for idx, r2_val in enumerate(r2_groups):
        row = idx // ncols
        col = idx % ncols
        ax = axes[row, col]
        sub_df = result_df[result_df['r2'] == r2_val].copy().sort_values('p_thresh')
        p_labels = [('%.0e' % p).replace('e', "E") for p in sub_df['p_thresh']]
        x = np.arange(len(sub_df))
        neg_log_p = -np.log10(sub_df['p_value'])
        bars = ax.bar(x, neg_log_p, color='coral', edgecolor='black')
        ax.set_xticks(x)
        ax.set_xticklabels(p_labels, rotation=45, ha='right', fontsize=8)
        ax.set_ylabel('-log10(P-value)')
        ax.set_ylim(0, 4)  # 可根据实际调整
        ax.set_title(f'R² = {r2_val}')
        ax.set_xlabel('P threshold')
        ax.axhline(y=-np.log10(0.05), color='red', linestyle='--', linewidth=0.8, label='P=0.05')
        ax.legend(fontsize=7)

        # 标注原始P值（科学计数法，两位有效数字）
        for bar, p_val in zip(bars, sub_df['p_value']):
            height = bar.get_height()
            if p_val < 0.05:
                p_label = f'{p_val:.2e}'
            else:
                p_label = f'{p_val:.2f}'
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.05,
                    p_label, ha='center', va='bottom', fontsize=10, rotation=45)

    for idx in range(n_groups, nrows*ncols):
        row = idx // ncols
        col = idx % ncols
        axes[row, col].axis('off')

    plt.tight_layout()
    plt.savefig(P_PLOT, dpi=300)
    plt.close()
    print(f"P值柱状图（-log10尺度，按R2分组）已保存至 {P_PLOT}")

    print("\n分析完成！")

if __name__ == "__main__":
    main()

加载连续表型数据...
加载了 765 个个体的表型值
找到 16 个分数文件
处理: all_PRS_result\clump_1000_r2_0.001\p_1e-05\sum_score_SPd18.txt
  完成: beta=-0.1812, p=2.85e-02, N=765
处理: all_PRS_result\clump_1000_r2_0.001\p_5e-06\sum_score_SPd18.txt
  完成: beta=-0.1812, p=2.85e-02, N=765
处理: all_PRS_result\clump_1000_r2_0.001\p_5e-07\sum_score_SPd18.txt
  完成: beta=-0.1792, p=4.03e-02, N=765
处理: all_PRS_result\clump_1000_r2_0.001\p_5e-08\sum_score_SPd18.txt
  完成: beta=-0.1792, p=4.03e-02, N=765
处理: all_PRS_result\clump_1000_r2_0.01\p_1e-05\sum_score_SPd18.txt
  完成: beta=-0.1812, p=2.85e-02, N=765
处理: all_PRS_result\clump_1000_r2_0.01\p_5e-06\sum_score_SPd18.txt
  完成: beta=-0.1812, p=2.85e-02, N=765
处理: all_PRS_result\clump_1000_r2_0.01\p_5e-07\sum_score_SPd18.txt
  完成: beta=-0.1792, p=4.03e-02, N=765
处理: all_PRS_result\clump_1000_r2_0.01\p_5e-08\sum_score_SPd18.txt
  完成: beta=-0.1792, p=4.03e-02, N=765
处理: all_PRS_result\clump_1000_r2_0.1\p_1e-05\sum_score_SPd18.txt
  完成: beta=-0.1297, p=6.97e-03, N=765
处理: all_PRS_result\cl